# Airline delay：原始数据清单 / Raw data inventory

本 Notebook 展示原始下载清单与覆盖。完整重建、各阶段运行及已有档案的核对方法见项目根目录的 `docs/最终运行说明.md`；实际运行状态以 `experiments/index.csv` 和各 run 的 manifest 为准。

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'raw/flights').is_dir(), 'Open this notebook from the project root or notebooks folder.'

## 1. 已下载范围 / Downloaded scope

月份统计与天气覆盖来自下载完整性核对，不是清洗后样本量。

In [ ]:
summary = json.loads((ROOT/'tables/download_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.read_csv(ROOT/'tables/flight_months.csv'))

## 2. 逐机场逐月天气覆盖 / Weather coverage

`rows` 是源档案记录或预报分段数；`days` 是有记录的不同 UTC 日期数。TAF 的一天会发布多个版本，每个版本又含多个预报分段。

In [ ]:
coverage = pd.read_csv(ROOT/'tables/weather_coverage.csv')
display(coverage.groupby(['kind','month']).agg(airports_with_records=('rows', lambda x: (x > 0).sum()), records=('rows','sum')).reset_index())
display(pd.read_csv(ROOT/'tables/weather_unavailable.csv'))

## 3. 文件迁移核对 / Offline integrity check

在终端运行 `python scripts/verify_downloads.py --raw-only`。检查哈希与压缩文件完整性，不联网、不改写原始数据。

航班清洗与衔接证据见 `01_航班清洗与衔接.ipynb`；开发比较及最终评估见 06、07 Notebook。迁移时同时同步并校验原始输入、本地派生数据、模型和预测，不能把 Git 克隆单独视为完整研究备份。